# Fine-tuning QLoRA — Assistente Virtual Médico (Tech Challenge Fase 3)

Notebook de execução do **issue #3** (Pessoa A). Roda no **Google Colab com GPU T4**
(`Ambiente de execução → Alterar tipo de ambiente de execução → T4 GPU`).

A lógica do treino **não vive aqui** — vive em `src/hospital_assistant/finetuning/train.py`.
Este notebook só orquestra: instala as dependências de GPU, prepara os dados, chama
`train()`, plota as curvas e publica o adapter. É o que atende o requisito de
"projeto modularizado em Python" do PDF sem transformar o notebook no código real.

**Configuração** (decisões fechadas em `docs/ESTRATEGIA.md` §1 e §3):

| item | valor |
|---|---|
| modelo base | `meta-llama/Llama-3.2-3B-Instruct` (fallback: espelho não-gated da Unsloth) |
| quantização | 4-bit NF4 + double quant, compute `float16` |
| LoRA | `r=16`, `alpha=32`, `dropout=0.05`, alvo `q_proj`/`v_proj` |
| treino | batch 4 × grad_accum 4, 3 epochs, `lr=2e-4` |

**Antes de rodar**, cadastre em `🔑 Secrets` (ícone de chave na barra lateral):
- `HF_TOKEN` — token da Hugging Face **com permissão de write** (necessário para o push do adapter no #4)
- `GOOGLE_API_KEY` **ou** `GROQ_API_KEY` — só se for gerar o dataset aqui (passo 2b)

## 1. Setup

In [1]:
# Confirma que a GPU está ativa antes de instalar 3GB de dependências.
!nvidia-smi

Fri Sep  4 14:54:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   53C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# Dependências de treino que não estão no ambiente padrão do projeto
# (extra `finetuning` do pyproject.toml — ver ESTRATEGIA.md §10).
!pip install -q -U transformers peft accelerate datasets huggingface_hub
!pip install -q -U bitsandbytes trl
!pip install -q python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 38.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 796.8/796.8 kB 43.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 992.6/992.6 kB 67.7 MB/s eta 0:00:00


In [3]:
# Clona o projeto e instala o pacote em modo editável, para que
# `import hospital_assistant` resolva o código real de src/.
import os

REPO = "https://github.com/fiap-postech-ia-para-devs-grupo/9IADT-fase-3-tech-challenge.git"
BRANCH = "feature/finetuning-2-preparacao-de-dados"  # troque para a branch do #3 enquanto o PR não foi mergeado

if not os.path.exists("9IADT-fase-3-tech-challenge"):
    !git clone --branch $BRANCH $REPO

%cd 9IADT-fase-3-tech-challenge
!pip install -q -e . --no-deps

import sys
sys.path.insert(0, "src")

Cloning into '9IADT-fase-3-tech-challenge'...
remote: Enumerating objects: 443, done.
remote: Counting objects: 100% (443/443), done.
remote: Compressing objects: 100% (310/310), done.
remote: Total 443 (delta 115), reused 400 (delta 83), pack-reused 0 (from 0)
Receiving objects: 100% (443/443), 1.01 MiB | 1.75 MiB/s, done.
Resolving deltas: 100% (115/115), done.
/content/9IADT-fase-3-tech-challenge
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
ERROR: Package 'hospital-assistant-fase3' requires a different Python: 3.13.15 not in '<3.13,>=3.11'


In [4]:
# Credenciais a partir dos Secrets do Colab.
from google.colab import userdata

for chave in ("HF_TOKEN", "GOOGLE_API_KEY", "GROQ_API_KEY"):
    try:
        os.environ[chave] = userdata.get(chave)
        print(f"{chave}: carregado")
    except Exception:
        print(f"{chave}: ausente (ok se não for usar)")

HF_TOKEN: carregado
GOOGLE_API_KEY: ausente (ok se não for usar)
GROQ_API_KEY: ausente (ok se não for usar)


In [5]:
# Checkpoints no Drive: a sessão do Colab cai no meio do treino
# (ESTRATEGIA.md §13, risco de probabilidade ALTA). Sem isso, recomeça do zero.
from google.colab import drive
drive.mount('/content/drive')

CHECKPOINT_DIR = "/content/drive/MyDrive/tech-challenge-fase3/adapter"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print("Checkpoints em:", CHECKPOINT_DIR)

Mounted at /content/drive
Checkpoints em: /content/drive/MyDrive/tech-challenge-fase3/adapter


## 2. Preparação de dados (#2)

`data/processed/` está no `.gitignore` (é artefato derivado), então o clone vem sem os
splits. Duas opções — **a** é a recomendada por ser reprodutível.

### 2a. Regenerar o dataset aqui

Baixa PubMedQA + MedQuAD, reaproveita o corpus sintético versionado em
`data/raw/sinteticos_finetuning.jsonl` (não gasta cota de API), anonimiza, cura,
deduplica e escreve `data/processed/{train,val}.jsonl`.

In [6]:
import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")

from hospital_assistant.finetuning.data_prep import prepare_dataset

train_ex, val_ex = prepare_dataset()
print(f"{len(train_ex)} treino / {len(val_ex)} validação")
train_ex[0]

README.md:   0%|          | 0.00/5.19k [00:00<?, ?B/s]

pqa_labeled/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.08MB            

pqa_labeled/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

README.md:   0%|          | 0.00/2.77k [00:00<?, ?B/s]

data/train-00000-of-00001-e36383d177026d(…): reconstructing file:   0%|          |  0.00B / 10.7MB            

data/train-00000-of-00001-e36383d177026d(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/47441 [00:00<?, ? examples/s]

869 treino / 97 validação


{'instruction': 'Is halofantrine ototoxic?',
 'input': 'Halofantrine is a newly developed antimalarial drug used for the treatment of Plasmodium falciparum malaria. The introduction of this drug has been delayed because of its possible side effects, and due to insufficient studies on adverse reactions in humans. There have been no studies investigating its effect on hearing. Thirty guinea pigs were divided into three groups: a control group, a halofantrine therapeutic dose group and a halofantrine double therapeutic dose group. One cochlea specimen from each animal was stained with haematoxylin and eosin and the other with toluidine blue. No changes were detected in the control group. The halofantrine therapeutic dose group showed loss and distortion of inner hair cells and inner phalangeal cells, and loss of spiral ganglia cells. In the halofantrine double therapeutic dose group, the inner and outer hair cells were distorted and there was loss of spiral ganglia cells.',
 'output': 'Ha

### 2b. (Alternativa) Subir os splits gerados na máquina local

Use se preferir treinar exatamente sobre o arquivo já revisado localmente.

In [ ]:
# from google.colab import files
# os.makedirs("data/processed", exist_ok=True)
# uploaded = files.upload()  # selecione train.jsonl e val.jsonl
# for nome in uploaded:
#     os.rename(nome, f"data/processed/{nome}")

In [7]:
# Conferência rápida da anonimização antes de gastar GPU: nenhum exemplo
# deve conter CPF, e-mail ou telefone reais.
import json, re

with open("data/processed/train.jsonl", encoding="utf-8") as f:
    exemplos = [json.loads(l) for l in f]

suspeitos = [
    e for e in exemplos
    if re.search(r"\d{3}\.\d{3}\.\d{3}-\d{2}|[\w.]+@[\w.]+\.\w+", json.dumps(e, ensure_ascii=False))
]
print(f"{len(exemplos)} exemplos | {len(suspeitos)} com PII residual")
assert not suspeitos, suspeitos[:3]

869 exemplos | 0 com PII residual


## 3. Fine-tuning QLoRA (#3)

Toda a configuração está em `train.py` (`LORA_KWARGS`, `TRAINING_KWARGS`) e é coberta
por testes de regressão em `tests/test_train.py` — não altere os valores aqui, altere lá.

Tempo esperado no T4: **~40-70 min** para ~900 exemplos × 3 epochs.

In [8]:
from pathlib import Path
from hospital_assistant.finetuning.train import train, LORA_KWARGS, TRAINING_KWARGS

print("LoRA:  ", LORA_KWARGS)
print("Treino:", TRAINING_KWARGS)

LoRA:   {'r': 16, 'lora_alpha': 32, 'lora_dropout': 0.05, 'target_modules': ['q_proj', 'v_proj'], 'task_type': 'CAUSAL_LM', 'bias': 'none'}
Treino: {'per_device_train_batch_size': 4, 'gradient_accumulation_steps': 4, 'num_train_epochs': 3, 'learning_rate': 0.0002, 'warmup_ratio': 0.03, 'lr_scheduler_type': 'cosine', 'logging_steps': 5, 'optim': 'paged_adamw_8bit', 'fp16': True, 'eval_strategy': 'epoch', 'save_strategy': 'epoch', 'save_total_limit': 2, 'report_to': 'none'}


In [12]:
# `resume_from_checkpoint=True` retoma do último checkpoint no Drive
# se a sessão tiver caído numa tentativa anterior.
metricas = train(output_dir=Path(CHECKPOINT_DIR), resume_from_checkpoint=False)
metricas["final_eval_perplexity"]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Applying formatting function to train dataset:   0%|          | 0/869 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/869 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/869 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/869 [00:00<?, ? examples/s]

...

Tokenizing eval dataset:   0%|          | 0/97 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/97 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/97 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/97 [00:00<?, ? examples/s]

NotImplementedError: "_amp_foreach_non_finite_check_and_unscale_cuda" not implemented for 'BFloat16'

In [10]:
import trl, inspect
from trl import SFTConfig
from hospital_assistant.finetuning.train import TRAINING_KWARGS
p = set(inspect.signature(SFTConfig.__init__).parameters)
print("trl", trl.__version__, "| params:", len(p))
print("REJEITADOS:", sorted(k for k in TRAINING_KWARGS if k not in p))
print("max_seq_length?", "max_seq_length" in p, "max_length?", "max_length" in p)

trl 1.12.0 | params: 134
REJEITADOS: ['warmup_ratio']
max_seq_length? False max_length? True


In [13]:
import transformers, torch, inspect
from transformers import AutoModelForCausalLM
sig = inspect.signature(AutoModelForCausalLM.from_pretrained).parameters
print("transformers", transformers.__version__, "| torch", torch.__version__)
print("dtype?", "dtype" in sig, "| torch_dtype?", "torch_dtype" in sig, "| aceita kwargs?", any(x.kind == x.VAR_KEYWORD for x in sig.values()))
print("bf16 nativo na GPU?", torch.cuda.is_bf16_supported())

transformers 5.16.1 | torch 2.11.0+cu128
dtype? False | torch_dtype? False | aceita kwargs? True
bf16 nativo na GPU? True


In [14]:
!git pull --ff-only
import importlib
from hospital_assistant.finetuning import train as train_mod
importlib.reload(train_mod)
print("recarregado")

remote: Enumerating objects: 25, done.
remote: Counting objects: 100% (25/25), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 16 (delta 12), reused 16 (delta 12), pack-reused 0 (from 0)
Unpacking objects: 100% (16/16), 72.13 KiB | 2.40 MiB/s, done.
From https://github.com/fiap-postech-ia-para-devs-grupo/9IADT-fase-3-tech-challenge
   3845528..96efd72  feature/finetuning-2-preparacao-de-dados -> origin/feature/finetuning-2-preparacao-de-dados
Updating 3845528..96efd72
Fast-forward
 pyproject.toml                             |   5 +-
 src/hospital_assistant/finetuning/train.py |  62 ++-
 tests/test_train.py                        |  92 ++++
 uv.lock                                    | 666 ++++++++++++++++++++++++++++-
 4 files changed, 800 insertions(+), 25 deletions(-)
recarregado


In [15]:
metricas = train_mod.train(output_dir=Path(CHECKPOINT_DIR), resume_from_checkpoint=False)
metricas["final_eval_perplexity"]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

NotImplementedError: "_amp_foreach_non_finite_check_and_unscale_cuda" not implemented for 'BFloat16'

In [11]:
from hospital_assistant.finetuning import train as train_mod
removidos = [train_mod.TRAINING_KWARGS.pop(k) for k in list(train_mod.TRAINING_KWARGS) if k not in p]
print("removidos:", removidos, "| restantes:", sorted(train_mod.TRAINING_KWARGS))

removidos: [0.03] | restantes: ['eval_strategy', 'fp16', 'gradient_accumulation_steps', 'learning_rate', 'logging_steps', 'lr_scheduler_type', 'num_train_epochs', 'optim', 'per_device_train_batch_size', 'report_to', 'save_strategy', 'save_total_limit']


## 4. Curvas de loss → `results/finetuning_metrics.json`

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot([p["epoch"] for p in metricas["train"]], [p["loss"] for p in metricas["train"]], label="treino")
if metricas["eval"]:
    ax.plot([p["epoch"] for p in metricas["eval"]], [p["loss"] for p in metricas["eval"]],
            marker="o", label="validação")
ax.set_xlabel("época"); ax.set_ylabel("loss"); ax.legend()
ax.set_title(f"QLoRA — perplexidade final: {metricas['final_eval_perplexity']:.2f}")
fig.savefig("results/loss_curve.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Publicação do adapter no Hugging Face Hub (#4)

Sobe **só o adapter LoRA** (alguns MB), nunca os pesos do modelo base — decisão de
ESTRATEGIA.md §1 e exigência do §12 ("pesos de modelo não commitados").

In [ ]:
from huggingface_hub import HfApi, whoami

usuario = whoami(token=os.environ["HF_TOKEN"])["name"]
ADAPTER_REPO = f"{usuario}/hospital-assistant-llama32-3b-lora"
print("Publicando em:", ADAPTER_REPO)

api = HfApi(token=os.environ["HF_TOKEN"])
api.create_repo(ADAPTER_REPO, repo_type="model", exist_ok=True, private=False)

# `allow_patterns` é essencial: CHECKPOINT_DIR também contém os
# `checkpoint-*/` do Trainer (estado do otimizador, scheduler, RNG — centenas
# de MB). Sem o filtro, o repositório público receberia tudo isso junto, o que
# contraria a decisão de publicar "só o adapter LoRA" (ESTRATEGIA.md §1).
api.upload_folder(
    folder_path=CHECKPOINT_DIR,
    repo_id=ADAPTER_REPO,
    repo_type="model",
    allow_patterns=[
        "adapter_config.json",
        "adapter_model.safetensors",
        "tokenizer*",
        "special_tokens_map.json",
        "chat_template.jinja",
        "README.md",
    ],
)
print(f"https://huggingface.co/{ADAPTER_REPO}")

# Conferência: o repositório deve conter apenas os arquivos do adapter.
print("
Arquivos publicados:")
for arquivo in api.list_repo_files(ADAPTER_REPO):
    print(" -", arquivo)

In [ ]:
# Guarde este valor: é o que o app lê para carregar o adapter em runtime.
# Coloque no .env do projeto:  HF_ADAPTER_REPO=<valor impresso abaixo>
print(f"HF_ADAPTER_REPO={ADAPTER_REPO}")

## 6. Avaliação base vs. fine-tuned (#4)

Roda as 9 perguntas clínicas de `evaluate.PERGUNTAS_AVALIACAO` nos dois modelos e grava
`results/eval_comparativo.json`. É o insumo da seção 3.3 do relatório técnico.

In [ ]:
os.environ["HF_ADAPTER_REPO"] = ADAPTER_REPO

from hospital_assistant.finetuning.evaluate import evaluate, resumir

linhas = evaluate()
resumir(linhas)

In [ ]:
for linha in linhas:
    print("=" * 100)
    print("PERGUNTA:  ", linha["question"])
    print("-" * 100)
    print("BASE:      ", linha["base_answer"][:600])
    print("-" * 100)
    print("FINE-TUNED:", linha["finetuned_answer"][:600])

## 7. Baixar os artefatos para commitar no repositório

`results/finetuning_metrics.json` e `results/eval_comparativo.json` são entregáveis
dos issues #3 e #4 e precisam entrar no Git (o adapter, não — ele vive no Hub).

In [ ]:
from google.colab import files

for artefato in ("results/finetuning_metrics.json", "results/eval_comparativo.json", "results/loss_curve.png"):
    if os.path.exists(artefato):
        files.download(artefato)

In [1]:
# Re-execucao completa numa VM nova, em celula unica: mantem o kernel ocupado
# do inicio ao fim, que e a defesa contra novo timeout por inatividade.
!pip install -q -U transformers peft accelerate datasets huggingface_hub
!pip install -q -U bitsandbytes trl
!git clone -q --branch feature/finetuning-2-preparacao-de-dados https://github.com/fiap-postech-ia-para-devs-grupo/9IADT-fase-3-tech-challenge.git /content/proj
import os, sys, logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
os.chdir("/content/proj")
sys.path.insert(0, "src")
from google.colab import userdata, drive
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
drive.mount("/content/drive")
CHECKPOINT_DIR = "/content/drive/MyDrive/tech-challenge-fase3/adapter"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print("=== setup ok, preparando dataset ===")
from pathlib import Path
from hospital_assistant.finetuning.data_prep import prepare_dataset
tr, vl = prepare_dataset()
print(f"=== dataset: {len(tr)} treino / {len(vl)} validacao — iniciando treino ===")
from hospital_assistant.finetuning.train import train
metricas = train(output_dir=Path(CHECKPOINT_DIR), resume_from_checkpoint=False)
print("=== TREINO CONCLUIDO — perplexidade final:", metricas["final_eval_perplexity"], "===")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 796.8/796.8 kB 51.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 992.6/992.6 kB 57.7 MB/s eta 0:00:00


ValueError: mount failed

In [2]:
# Sem Google Drive: o mount falhou duas vezes e so servia para resiliencia a
# queda de sessao. O destino do adapter e o HF Hub, nao o Drive.
import os, sys, logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
os.chdir("/content/proj")
sys.path.insert(0, "src")
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
CHECKPOINT_DIR = "/content/proj/outputs/adapter"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
from pathlib import Path
from hospital_assistant.finetuning.data_prep import prepare_dataset
tr, vl = prepare_dataset()
print(f"=== dataset: {len(tr)} treino / {len(vl)} validacao — iniciando treino ===")
from hospital_assistant.finetuning.train import train
metricas = train(output_dir=Path(CHECKPOINT_DIR), resume_from_checkpoint=False)
print("=== TREINO CONCLUIDO — perplexidade final:", metricas["final_eval_perplexity"], "===")

README.md:   0%|          | 0.00/5.19k [00:00<?, ?B/s]

pqa_labeled/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.08MB            

pqa_labeled/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

README.md:   0%|          | 0.00/2.77k [00:00<?, ?B/s]

...

Tokenizing eval dataset:   0%|          | 0/97 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/97 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/97 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/97 [00:00<?, ? examples/s]

NotImplementedError: "_amp_foreach_non_finite_check_and_unscale_cuda" not implemented for 'BFloat16'

In [3]:
import os, sys, logging, importlib
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
os.chdir("/content/proj")
!git pull --ff-only
sys.path.insert(0, "src")
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
CHECKPOINT_DIR = "/content/proj/outputs/adapter"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
from pathlib import Path
import hospital_assistant.finetuning.train as tm
importlib.reload(tm)
from hospital_assistant.finetuning.data_prep import prepare_dataset
tr, vl = prepare_dataset()
print(f"=== dataset {len(tr)}/{len(vl)} — iniciando treino ===")
metricas = tm.train(output_dir=Path(CHECKPOINT_DIR), resume_from_checkpoint=False)
print("=== TREINO CONCLUIDO — perplexidade final:", metricas["final_eval_perplexity"], "===")

remote: Enumerating objects: 21, done.
remote: Counting objects: 100% (21/21), done.
remote: Compressing objects: 100% (6/6), done.
remote: Total 13 (delta 6), reused 13 (delta 6), pack-reused 0 (from 0)
Unpacking objects: 100% (13/13), 443.12 KiB | 3.79 MiB/s, done.
From https://github.com/fiap-postech-ia-para-devs-grupo/9IADT-fase-3-tech-challenge
   96efd72..b06733e  feature/finetuning-2-preparacao-de-dados -> origin/feature/finetuning-2-preparacao-de-dados
Updating 96efd72..b06733e
Fast-forward
 .gitignore                                 |   6 +
 data/processed/train.jsonl                 | 869 +++++++++++++++++++++++++++++
 data/processed/val.jsonl                   |  97 ++++
 src/hospital_assistant/finetuning/train.py |  32 +-
 4 files changed, 990 insertions(+), 14 deletions(-)
 create mode 100644 data/processed/train.jsonl
 create mode 100644 data/processed/val.jsonl


=== dataset 869/97 — iniciando treino ===


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

...

Building labels for eval dataset:   0%|          | 0/97 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/97 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/97 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,1.230392,1.249774,1.273584,408746.000000,0.722867
2,1.132663,1.208548,1.238296,817492.000000,0.730709
3,1.168185,1.200840,1.219431,1226238.000000,0.732437


=== TREINO CONCLUIDO — perplexidade final: 3.3229053959629473 ===


In [1]:
# Execucao definitiva: o artefato SAI da VM no mesmo instante em que existe.
# Na tentativa anterior o treino concluiu, mas a VM foi reciclada durante a
# janela ociosa seguinte e levou o adapter junto.
import os, sys, json, logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
!pip install -q -U transformers peft accelerate datasets huggingface_hub bitsandbytes trl
!git clone -q --branch feature/finetuning-2-preparacao-de-dados https://github.com/fiap-postech-ia-para-devs-grupo/9IADT-fase-3-tech-challenge.git /content/final
os.chdir("/content/final")
sys.path.insert(0, "src")
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
from pathlib import Path
from hospital_assistant.finetuning.data_prep import prepare_dataset
tr, vl = prepare_dataset()
print(f"=== dataset {len(tr)}/{len(vl)} — treinando ===")
from hospital_assistant.finetuning.train import train
ADAPTER_DIR = Path("/content/final/outputs/adapter")
metricas = train(output_dir=ADAPTER_DIR, resume_from_checkpoint=False)
print("=== PERPLEXIDADE FINAL:", metricas["final_eval_perplexity"], "===")
from huggingface_hub import HfApi, whoami
usuario = whoami(token=os.environ["HF_TOKEN"])["name"]
ADAPTER_REPO = f"{usuario}/hospital-assistant-llama32-3b-lora"
api = HfApi(token=os.environ["HF_TOKEN"])
api.create_repo(ADAPTER_REPO, repo_type="model", exist_ok=True, private=False)
api.upload_folder(folder_path=str(ADAPTER_DIR), repo_id=ADAPTER_REPO, repo_type="model", allow_patterns=["adapter_config.json", "adapter_model.safetensors", "tokenizer*", "special_tokens_map.json", "chat_template.jinja"])
print("=== ADAPTER PUBLICADO:", ADAPTER_REPO, "===")
print("=== METRICS JSON (copiar para results/finetuning_metrics.json) ===")
print(json.dumps(metricas, ensure_ascii=False, indent=2, default=str))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 796.8/796.8 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 992.6/992.6 kB 41.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 13.3 MB/s eta 0:00:00


README.md:   0%|          | 0.00/5.19k [00:00<?, ?B/s]

pqa_labeled/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.08MB            

pqa_labeled/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

...

Building labels for eval dataset:   0%|          | 0/97 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/97 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/97 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,1.230259,1.250400,1.283473,408746.000000,0.721526
2,1.133616,1.208370,1.236093,817492.000000,0.731231
3,1.169635,1.200766,1.219623,1226238.000000,0.732320


=== PERPLEXIDADE FINAL: 3.3226625825444986 ===
=== ADAPTER PUBLICADO: agendesse/hospital-assistant-llama32-3b-lora ===
=== METRICS JSON (copiar para results/finetuning_metrics.json) ===
{
  "train": [
    {
      "step": 5,
      "epoch": 0.09174311926605505,
      "loss": 2.483047294616699
    },
    {
      "step": 10,
      "epoch": 0.1834862385321101,
      "loss": 2.175557327270508
    },
    {
      "step": 15,
      "epoch": 0.27522935779816515,
      "loss": 1.8655204772949219
    },
    {
      "step": 20,
      "epoch": 0.3669724770642202,
      "loss": 1.5757454872131347
    },
    {
      "step": 25,
      "epoch": 0.45871559633027525,
      "loss": 1.4525952339172363
    },
    {
      "step": 30,
      "epoch": 0.5504587155963303,
      "loss": 1.3031782150268554
    },
    {
      "step": 35,
      "epoch": 0.6422018348623854,
      "loss": 1.3167473793029785
    },
    {
      "step": 40,
      "epoch": 0.7339449541284404,
      "loss": 1.2942416191101074
    },
    {
 

In [2]:
# Sobe as metricas para o Hub tambem, para recupera-las com exatidao mesmo se a VM cair.
from huggingface_hub import HfApi
api = HfApi(token=os.environ["HF_TOKEN"])
api.upload_file(path_or_fileobj="results/finetuning_metrics.json", path_in_repo="finetuning_metrics.json", repo_id=ADAPTER_REPO, repo_type="model")
api.upload_file(path_or_fileobj="results/dataset_stats.json", path_in_repo="dataset_stats.json", repo_id=ADAPTER_REPO, repo_type="model")
print("=== METRICAS NO HUB ===")
print(api.list_repo_files(ADAPTER_REPO))

=== METRICAS NO HUB ===
['.gitattributes', 'adapter_config.json', 'adapter_model.safetensors', 'chat_template.jinja', 'dataset_stats.json', 'finetuning_metrics.json', 'tokenizer.json', 'tokenizer_config.json']


In [3]:
# Comparativo base vs. fine-tuned (#4). Carrega os dois modelos em 4-bit.
os.environ["HF_ADAPTER_REPO"] = ADAPTER_REPO
import importlib, hospital_assistant.finetuning.evaluate as ev
importlib.reload(ev)
linhas = ev.evaluate()
resumo = ev.resumir(linhas)
print("=== RESUMO ===")
print(json.dumps(resumo, ensure_ascii=False, indent=2))
api.upload_file(path_or_fileobj="results/eval_comparativo.json", path_in_repo="eval_comparativo.json", repo_id=ADAPTER_REPO, repo_type="model")
print("=== COMPARATIVO NO HUB ===")

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

adapter_config.json:   0%|          | 0.00/1.08k [00:00<?, ?B/s]

adapter_model.safetensors: reconstructing file:   0%|          |  0.00B / 18.4MB            

adapter_model.safetensors: downloading bytes:           |  0.00B            

=== RESUMO ===
{
  "perguntas": 9,
  "tamanho_medio_base": 24,
  "tamanho_medio_finetuned": 24,
  "respostas_que_exigem_validacao_base": 0,
  "respostas_que_exigem_validacao_finetuned": 0
}
=== COMPARATIVO NO HUB ===


In [4]:
!git pull --ff-only
import importlib
import hospital_assistant.llm.model_loader as ml
import hospital_assistant.finetuning.evaluate as ev
importlib.reload(ml)
importlib.reload(ev)
ml.get_llm.cache_clear()
linhas = ev.evaluate()
resumo = ev.resumir(linhas)
print("=== RESUMO ===")
print(json.dumps(resumo, ensure_ascii=False, indent=2))
print("=== AMOSTRA ===")
print("BASE:", linhas[0]["base_answer"][:300])
print("TUNED:", linhas[0]["finetuned_answer"][:300])
api.upload_file(path_or_fileobj="results/eval_comparativo.json", path_in_repo="eval_comparativo.json", repo_id=ADAPTER_REPO, repo_type="model")
print("=== NO HUB ===")

remote: Enumerating objects: 26, done.
remote: Counting objects: 100% (26/26), done.
remote: Compressing objects: 100% (6/6), done.
remote: Total 16 (delta 9), reused 16 (delta 9), pack-reused 0 (from 0)
Unpacking objects: 100% (16/16), 6.28 KiB | 919.00 KiB/s, done.
From https://github.com/fiap-postech-ia-para-devs-grupo/9IADT-fase-3-tech-challenge
   b06733e..a6046ad  feature/finetuning-2-preparacao-de-dados -> origin/feature/finetuning-2-preparacao-de-dados
Updating b06733e..a6046ad
error: The following untracked working tree files would be overwritten by merge:
	results/eval_comparativo.json
	results/finetuning_metrics.json
Please move or remove them before you merge.
Aborting


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

=== RESUMO ===
{
  "perguntas": 9,
  "tamanho_medio_base": 24,
  "tamanho_medio_finetuned": 24,
  "respostas_que_exigem_validacao_base": 0,
  "respostas_que_exigem_validacao_finetuned": 0
}
=== AMOSTRA ===
BASE: [ERRO: AttributeError: ]
TUNED: [ERRO: AttributeError: ]


No files have been modified since last commit. Skipping to prevent empty commit.


=== NO HUB ===


In [5]:
print(os.getcwd())
!git -C /content/final log --oneline -1
!grep -c return_dict /content/final/src/hospital_assistant/finetuning/evaluate.py
!grep -c return_dict /content/final/src/hospital_assistant/llm/model_loader.py
import inspect, hospital_assistant.finetuning.evaluate as ev2
print("repr no fonte carregado?", "erro!r" in inspect.getsource(ev2.comparar))

/content/final
b06733e (HEAD -> feature/finetuning-2-preparacao-de-dados) fix(finetuning): converter os parâmetros LoRA para fp32, não o modelo base (#3)
0
0
repr no fonte carregado? False


In [7]:
from google.colab.output import eval_js
print(eval_js("google.colab.kernel.p  roxyPort(8501)"))

https://8501-gpu-t4-s-kkb-euw4c1-36fknlopv1jk1-c.europe-west4-1.prod.colab.dev


In [6]:
!git -C /content/final fetch origin feature/finetuning-2-preparacao-de-dados
!git -C /content/final reset --hard origin/feature/finetuning-2-preparacao-de-dados
!git -C /content/final log --oneline -1
!grep -c return_dict /content/final/src/hospital_assistant/finetuning/evaluate.py

From https://github.com/fiap-postech-ia-para-devs-grupo/9IADT-fase-3-tech-challenge
 * branch            feature/finetuning-2-preparacao-de-dados -> FETCH_HEAD
HEAD is now at a6046ad fix(llm): adaptar a geração ao BatchEncoding do transformers 5 (#4)
a6046ad (HEAD -> feature/finetuning-2-preparacao-de-dados, origin/feature/finetuning-2-preparacao-de-dados) fix(llm): adaptar a geração ao BatchEncoding do transformers 5 (#4)
2


In [8]:
!tail -n 12 /content/portal.log

2026-09-05 02:58:04.688 Rejecting WebSocket connection with disallowed Origin or Host header: origin=https://8501-gpu-t4-s-kkb-euw4c1-36fknlopv1jk1-c.europe-west4-1.prod.colab.dev, host=gpu-t4-s-kkb-euw4c1-36fknlopv1jk1.europe-west4-c.c.codatalab-user-runtimes.internal:8007
2026-09-05 02:58:11.876 Rejecting WebSocket connection with disallowed Origin or Host header: origin=https://8501-gpu-t4-s-kkb-euw4c1-36fknlopv1jk1-c.europe-west4-1.prod.colab.dev, host=gpu-t4-s-kkb-euw4c1-36fknlopv1jk1.europe-west4-c.c.codatalab-user-runtimes.internal:8007
2026-09-05 02:58:21.128 Rejecting WebSocket connection with disallowed Origin or Host header: origin=https://8501-gpu-t4-s-kkb-euw4c1-36fknlopv1jk1-c.europe-west4-1.prod.colab.dev, host=gpu-t4-s-kkb-euw4c1-36fknlopv1jk1.europe-west4-c.c.codatalab-user-runtimes.internal:8007
2026-09-05 02:58:29.731 Rejecting WebSocket connection with disallowed Origin or Host header: origin=https://8501-gpu-t4-s-kkb-euw4c1-36fknlopv1jk1-c.europe-west4-1.prod.colab

In [10]:
!curl -sL https://raw.githubusercontent.com/fiap-postech-ia-para-devs-grupo/9IADT-fase-3-tech-challenge/main/scripts/colab_portal.sh | bash

==> 1/5  Código (branch main)
==> 2/5  Dependências
==> 3/5  Dados (SQLite de pacientes e índice vetorial)
/content/portal/src/hospital_assistant/rag/ingest.py:13: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import HuggingFaceEmbeddings
/content/portal/src/hospital_assistant/rag/ingest.py:78: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL, **EMBEDDING_KWARGS)
Loading weights: 100% 103/103 

In [7]:
import sys
_ = [sys.modules.pop(k) for k in [x for x in list(sys.modules) if x.startswith("hospital_assistant")]]
from hospital_assistant.finetuning.evaluate import evaluate as _eval, resumir as _resumir
linhas = _eval()
resumo = _resumir(linhas)
print("=== RESUMO ===")
print(json.dumps(resumo, ensure_ascii=False, indent=2))
print("=== AMOSTRA P1 ===")
print("BASE  :", linhas[0]["base_answer"][:400])
print("TUNED :", linhas[0]["finetuned_answer"][:400])
api.upload_file(path_or_fileobj="results/eval_comparativo.json", path_in_repo="eval_comparativo.json", repo_id=ADAPTER_REPO, repo_type="model")
print("=== NO HUB ===")

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https:/

=== RESUMO ===
{
  "perguntas": 9,
  "tamanho_medio_base": 1391,
  "tamanho_medio_finetuned": 560,
  "respostas_que_exigem_validacao_base": 3,
  "respostas_que_exigem_validacao_finetuned": 4
}
=== AMOSTRA P1 ===
BASE  : Para um paciente com suspeita de sepse, a conduta inicial recomendada é:

**Observação intensiva**

O paciente deve ser observado em uma unidade de observação intensiva (UI) ou unidade de cuidados intensivos (UCI), onde possa ser monitorado de perto e receber cuidados de apoio adequados.

**Avaliação clínica**

O médico responsável deve realizar uma avaliação clínica completa, incluindo:

* Exame 
TUNED : Sugiro iniciar a terapia intravenosa com solução de salina 0,9 % e adicionar antibiótico de amplo espectro, como ceftriaxona 2 g, administrado via veia central, após obter cultura sanguínea. Recomendo também iniciar a terapia de oxigênio e monitorar a pressão arterial. O médico responsável deve validar a prescrição e assinar o formulário de prescrição antes de iniciar 